In [ ]:
!pip install -q -U transformers datasets accelerate

In [ ]:
from datasets import load_dataset

ds = load_dataset("AI-Lab-Makerere/beans")
etiquetas = ds["train"].features["labels"].names
print(ds)
print(etiquetas)
ds["train"][0]["image"]

In [ ]:
import numpy as np
import torch
from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                          TrainingArguments, Trainer)

base = "google/vit-base-patch16-224-in21k"
procesador = AutoImageProcessor.from_pretrained(base)


def transformar(lote):
    entradas = procesador([img.convert("RGB") for img in lote["image"]], return_tensors="pt")
    entradas["labels"] = lote["labels"]
    return entradas


ds_listo = ds.with_transform(transformar)


def juntar(lote):
    return {"pixel_values": torch.stack([x["pixel_values"] for x in lote]),
            "labels": torch.tensor([x["labels"] for x in lote])}


def metricas(p):
    return {"exactitud": (np.argmax(p.predictions, axis=1) == p.label_ids).mean()}


modelo = AutoModelForImageClassification.from_pretrained(
    base, num_labels=len(etiquetas),
    id2label=dict(enumerate(etiquetas)),
    label2id={e: i for i, e in enumerate(etiquetas)})

args = TrainingArguments(
    output_dir="vit-frijol",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
)

entrenador = Trainer(model=modelo, args=args,
                     train_dataset=ds_listo["train"],
                     eval_dataset=ds_listo["validation"],
                     data_collator=juntar, compute_metrics=metricas)
entrenador.train()

In [ ]:
print(entrenador.evaluate(ds_listo["test"]))

In [ ]:
from transformers import pipeline

clasificador = pipeline("image-classification", model=entrenador.model,
                        image_processor=procesador, device=0)
clasificador(ds["test"][5]["image"])

In [ ]:
# E.5 (publicar) se prueba aparte: requiere HF_TOKEN